# Chapter 10 - Abstract Base Classes, Protocols, and Duck Typing

#### 1. Mechanical Refresher
An abstract base class uses `abc.ABC` and `@abstractmethod` to state that subclasses must provide specific methods before they can be instantiated. A `typing.Protocol` states a structural contract: an object fits if it has the required methods or attributes, even if it does not inherit from the protocol.

#### 2. Minimal Working Example

In [ ]:
from abc import ABC, abstractmethod

class Scorer(ABC):
    @abstractmethod
    def score(self, values: list[float]) -> float:
        pass

class MeanScorer(Scorer):
    def score(self, values: list[float]) -> float:
        return sum(values) / len(values)

scorer: Scorer = MeanScorer()
assert scorer.score([2.0, 4.0]) == 3.0
print("checks passed")

The base class cannot be used directly because `score` is abstract. The subclass becomes concrete by defining `score` with the expected method name and behavior.

#### 3. Modify Drills

**Modify Drill 1.** Add a second concrete scorer and verify both share the base interface.

In [ ]:
from abc import ABC, abstractmethod

class Scorer(ABC):
    @abstractmethod
    def score(self, values: list[float]) -> float:
        pass

class MaxScorer(Scorer):
    def score(self, values: list[float]) -> float:
        return max(values)

scorer: Scorer = MaxScorer()
assert scorer.score([1.0, 5.0, 2.0]) == 5.0
print("passed")

**Modify Drill 2.** Use a protocol for any object with `predict`.

In [ ]:
from typing import Protocol

class Predictor(Protocol):
    def predict(self, value: float) -> float:
        ...

class Doubler:
    def predict(self, value: float) -> float:
        return value * 2

model: Predictor = Doubler()
assert model.predict(3.0) == 6.0
print("passed")

**Modify Drill 3.** Demonstrate duck typing with no inheritance.

In [ ]:
def run_predictor(model, value: float) -> float:
    return model.predict(value)

class OffsetModel:
    def predict(self, value: float) -> float:
        return value + 1

assert run_predictor(OffsetModel(), 4.0) == 5.0
print("passed")

#### 4. Break-and-Fix Drills

**Break-and-Fix Drill 1.** Break it by renaming `score` to `scores`. Predict why the subclass remains abstract, then restore the exact method name.

In [ ]:
from abc import ABC, abstractmethod

class Metric(ABC):
    @abstractmethod
    def score(self, values: list[float]) -> float:
        pass

class CountMetric(Metric):
    def score(self, values: list[float]) -> float:
        return float(len(values))

metric = CountMetric()
assert metric.score([1.0, 2.0]) == 2.0
print("passed")

**Break-and-Fix Drill 2.** Break protocol compatibility by removing `predict`. Predict which call fails, then restore the method.

In [ ]:
from typing import Protocol

class Predictor(Protocol):
    def predict(self, value: float) -> float:
        ...

class Identity:
    def predict(self, value: float) -> float:
        return value

def use_predictor(model: Predictor) -> float:
    return model.predict(2.0)

assert use_predictor(Identity()) == 2.0
print("passed")

#### 5. Self-Verification
Use asserts on behavior and, for ABCs, confirm concrete subclasses can be instantiated. Protocol checks are mostly structural: if the function can call the required method and return the expected value, the object satisfies the practical contract.

#### 6. Standalone Exercises

**Exercise 1.** Fill in a concrete subclass of an abstract transformer. Expected behavior: `6`.

In [ ]:
from abc import ABC, abstractmethod

class Transformer(ABC):
    @abstractmethod
    def transform(self, value: int) -> int:
        pass

class TimesTwo(Transformer):
    def transform(self, value: int) -> int:
        return 0  # TODO

assert TimesTwo().transform(3) == 6
print("passed")

**Exercise 2.** Define a protocol for objects with `label()`. Expected behavior: `'ok'`.

In [ ]:
from typing import Protocol

class HasLabel(Protocol):
    # TODO: declare label(self) -> str.
    pass

class Status:
    def label(self) -> str:
        return "ok"

item: HasLabel = Status()
assert "label" in HasLabel.__dict__
assert item.label() == "ok"
print("passed")

**Exercise 3.** Write a function that accepts the protocol. Expected behavior: `'item:x'`.

In [ ]:
from typing import Protocol

class Named(Protocol):
    name: str

class Item:
    name = "x"

def describe(obj: Named) -> str:
    return ""  # TODO

assert describe(Item()) == "item:x"
print("passed")

**Exercise 4.** Use an ABC for a report interface. Expected behavior: `'report'`.

In [ ]:
from abc import ABC, abstractmethod

class Report(ABC):
    @abstractmethod
    def render(self) -> str:
        pass

class TextReport(Report):
    def render(self) -> str:
        return ""  # TODO

assert TextReport().render() == "report"
print("passed")

**Exercise 5.** Use duck typing with two unrelated classes. Expected behavior: `[2, 3]`.

In [ ]:
class PlusOne:
    def predict(self, x: int) -> int:
        return x + 1

class PlusTwo:
    def predict(self, x: int) -> int:
        return x + 2

def run_all(models, x: int) -> list[int]:
    return []  # TODO

assert run_all([PlusOne(), PlusTwo()], 1) == [2, 3]
print("passed")

#### 7. Applied AI/ML Drill
**ML to Python mirror:** a minimal model ABC with `forward` or `predict` is just an interface requiring subclasses to supply those methods. **Python to ML mirror:** this is the same pattern behind `nn.Module`-style subclassing: PyTorch adds a lot of bookkeeping, but the visible contract is still "subclass this base and implement the expected call method."

**Applied Drill.** Complete a minimal model ABC and concrete linear model. Expected behavior: prediction `7.0`.

In [ ]:
from abc import ABC, abstractmethod

class Model(ABC):
    @abstractmethod
    def predict(self, x: float) -> float:
        pass

class LinearModel(Model):
    def __init__(self, weight: float, bias: float):
        self.weight = weight
        self.bias = bias

    def predict(self, x: float) -> float:
        return 0.0  # TODO

model: Model = LinearModel(2.0, 1.0)
assert model.predict(3.0) == 7.0
print("passed")

#### 8. Common Bugs
- Method names do not match exactly: the symptom is a subclass that still cannot be instantiated.
- Treating protocols as inheritance requirements: protocols are about shape unless you deliberately use runtime checks.
- Making an ABC too broad: the symptom is subclasses forced to implement methods they do not naturally support.
- Confusing duck typing with no contract: the contract still exists; it is just enforced by the operations your code performs.

#### 9. Compounding Drill

Combine protocols with dataclasses: create a typed config object and a model that satisfies a protocol without inheriting from it.

In [ ]:
from dataclasses import dataclass
from typing import Protocol

@dataclass
class ScaleConfig:
    factor: float

class Predictor(Protocol):
    def predict(self, x: float) -> float:
        ...

class Scaler:
    def __init__(self, config: ScaleConfig):
        self.config = config

    def predict(self, x: float) -> float:
        return 0.0  # TODO

model: Predictor = Scaler(ScaleConfig(3.0))
assert model.predict(2.0) == 6.0
print("passed")

#### 10. Chapter Project
No chapter project in this chapter. Projects appear every 2-3 chapters so the longer drills can compound several mechanics at once.

#### 11. Solutions Appendix

--- SOLUTIONS: DO NOT PEEK UNTIL ATTEMPTED ---

In [ ]:
# Standalone Exercises 1-3
from abc import ABC, abstractmethod
from typing import Protocol

class Transformer(ABC):
    @abstractmethod
    def transform(self, value: int) -> int:
        pass

class TimesTwo(Transformer):
    def transform(self, value: int) -> int:
        return value * 2

assert TimesTwo().transform(3) == 6

class HasLabel(Protocol):
    def label(self) -> str:
        ...

class Status:
    def label(self) -> str:
        return "ok"

item: HasLabel = Status()
assert item.label() == "ok"

class Named(Protocol):
    name: str

class Item:
    name = "x"

def describe(obj: Named) -> str:
    return "item:" + obj.name

assert describe(Item()) == "item:x"
print("solutions passed")

In [ ]:
# Standalone Exercises 4-5, Applied Drill, and Compounding Drill
from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import Protocol

class Report(ABC):
    @abstractmethod
    def render(self) -> str:
        pass

class TextReport(Report):
    def render(self) -> str:
        return "report"

assert TextReport().render() == "report"

class PlusOne:
    def predict(self, x: int) -> int:
        return x + 1

class PlusTwo:
    def predict(self, x: int) -> int:
        return x + 2

def run_all(models, x: int) -> list[int]:
    return [model.predict(x) for model in models]

assert run_all([PlusOne(), PlusTwo()], 1) == [2, 3]

class Model(ABC):
    @abstractmethod
    def predict(self, x: float) -> float:
        pass

class LinearModel(Model):
    def __init__(self, weight: float, bias: float):
        self.weight = weight
        self.bias = bias

    def predict(self, x: float) -> float:
        return self.weight * x + self.bias

assert LinearModel(2.0, 1.0).predict(3.0) == 7.0

@dataclass
class ScaleConfig:
    factor: float

class Predictor(Protocol):
    def predict(self, x: float) -> float:
        ...

class Scaler:
    def __init__(self, config: ScaleConfig):
        self.config = config

    def predict(self, x: float) -> float:
        return x * self.config.factor

model: Predictor = Scaler(ScaleConfig(3.0))
assert model.predict(2.0) == 6.0
print("solutions passed")